# Week 6 Final Data Analytics Project: HR Analytics
**Note:** This educational project uses a synthetic dataset. It does not represent a real company or real employees.

## Objective
Inspect and clean HR data, explore attrition patterns, visualize key metrics, and prepare a cleaned CSV for Power BI.

In [ ]:
# Week 6 Final Data Analytics Project: HR Analytics
# Dataset is synthetic and generated for educational demonstration.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("hr_analytics_dataset.csv")
print("Raw shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("Data type"))
display(df.isna().sum().to_frame("Missing values"))
print("Duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

# Clean data: remove exact duplicates; impute numeric missing values with medians
df_clean = df.drop_duplicates().copy()
for col in ["MonthlyIncome", "JobSatisfaction"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
df_clean["MonthlyIncome"] = df_clean["MonthlyIncome"].round().astype(int)
df_clean["JobSatisfaction"] = df_clean["JobSatisfaction"].round().astype(int)
df_clean["AttritionFlag"] = (df_clean["Attrition"] == "Yes").astype(int)
df_clean["OverTimeFlag"] = (df_clean["OverTime"] == "Yes").astype(int)

print("Clean shape:", df_clean.shape)
print("Missing values after cleaning:", int(df_clean.isna().sum().sum()))
print("Duplicates after cleaning:", int(df_clean.duplicated().sum()))
display(df_clean.describe())

# KPIs
total_employees = df_clean["EmployeeID"].nunique()
attrition_count = int(df_clean["AttritionFlag"].sum())
attrition_rate = 100 * attrition_count / total_employees
avg_monthly_income = df_clean["MonthlyIncome"].mean()
avg_tenure = df_clean["YearsAtCompany"].mean()
print(f"Employees: {total_employees}")
print(f"Employees left: {attrition_count}")
print(f"Attrition rate: {attrition_rate:.2f}%")
print(f"Average monthly income: {avg_monthly_income:,.0f}")
print(f"Average tenure: {avg_tenure:.2f} years")

# EDA: categorical summaries
display(df_clean["Department"].value_counts().to_frame("Employee count"))
display(df_clean.groupby("Department").agg(
    Employees=("EmployeeID", "nunique"),
    Attrition_Rate=("AttritionFlag", "mean"),
    Average_Monthly_Income=("MonthlyIncome", "mean"),
    Average_Tenure=("YearsAtCompany", "mean")
).assign(Attrition_Rate=lambda x: 100*x["Attrition_Rate"]).round(2))

# Correlation analysis (numeric variables only)
numeric_cols = ["Age", "MonthlyIncome", "YearsAtCompany", "JobSatisfaction",
                "WorkLifeBalance", "PerformanceRating", "AttritionFlag", "OverTimeFlag"]
corr = df_clean[numeric_cols].corr(numeric_only=True)
plt.figure(figsize=(9,6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# Visualization 1: attrition rate by department
dept_rate = df_clean.groupby("Department")["AttritionFlag"].mean().mul(100).sort_values(ascending=False)
dept_rate.plot(kind="bar", figsize=(8,5), title="Attrition Rate by Department")
plt.ylabel("Attrition rate (%)"); plt.xlabel("Department"); plt.tight_layout(); plt.show()

# Visualization 2: attrition by overtime
(df_clean.groupby("OverTime")["AttritionFlag"].mean().reindex(["No","Yes"]).mul(100)
 .plot(kind="bar", figsize=(6,4), title="Attrition Rate by Overtime"))
plt.ylabel("Attrition rate (%)"); plt.xlabel("Overtime"); plt.tight_layout(); plt.show()

# Visualization 3: average monthly income by department
df_clean.groupby("Department")["MonthlyIncome"].mean().sort_values().plot(
    kind="bar", figsize=(8,5), title="Average Monthly Income by Department")
plt.ylabel("Average monthly income"); plt.xlabel("Department"); plt.tight_layout(); plt.show()

# Visualization 4: satisfaction distribution
df_clean["JobSatisfaction"].value_counts().sort_index().plot(
    kind="bar", figsize=(6,4), title="Job Satisfaction Distribution")
plt.ylabel("Employee count"); plt.xlabel("Job satisfaction score (1–4)"); plt.tight_layout(); plt.show()

# Visualization 5: tenure by attrition status
df_clean.groupby("Attrition")["YearsAtCompany"].mean().reindex(["No","Yes"]).plot(
    kind="bar", figsize=(6,4), title="Average Tenure by Attrition Status")
plt.ylabel("Average years at company"); plt.xlabel("Attrition"); plt.tight_layout(); plt.show()

# Visualization 6: age versus monthly income
plt.figure(figsize=(8,5))
sns.scatterplot(data=df_clean, x="Age", y="MonthlyIncome", hue="Attrition", alpha=.7)
plt.title("Age vs Monthly Income by Attrition"); plt.tight_layout(); plt.show()

# Outlier check using IQR on MonthlyIncome
q1, q3 = df_clean["MonthlyIncome"].quantile([.25, .75])
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
income_outliers = df_clean[df_clean["MonthlyIncome"].lt(lower) | df_clean["MonthlyIncome"].gt(upper)]
print("Monthly income IQR bounds:", round(lower, 2), round(upper, 2))
print("Potential income outliers:", len(income_outliers))
# Outliers should be investigated, not automatically removed.

# Export clean data for Power BI
df_clean.to_csv("hr_analytics_cleaned.csv", index=False)
print("Exported hr_analytics_cleaned.csv")
